# Agentic RAG System – AgentEval Evaluation

## Project Objective
Build and evaluate an Agentic RAG baseline that plans retrieval steps, selects helper tools, retrieves evidence from a knowledge base, and returns answers with source chunk IDs.

This notebook focuses on the AgentEval benchmark. The companion Streamlit application supports document ingestion from PDF, DOCX, TXT, and Markdown files.

## 1. Install and import libraries

In [1]:
!pip -q install pandas numpy scikit-learn

In [2]:
import os
import re
import ast
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 160)

## 2. Load the AgentEval dataset

In [3]:
DATA_DIR = "/kaggle/input/competitions/agent-eval-part-i-grounded-rag-benchmark"

corpus = pd.read_csv(f"{DATA_DIR}/corpus.csv").fillna("")
train = pd.read_csv(f"{DATA_DIR}/train.csv").fillna("")
test = pd.read_csv(f"{DATA_DIR}/test.csv").fillna("")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("Corpus:", corpus.shape)
print("Train:", train.shape)
print("Test:", test.shape)
print("Sample submission:", sample_submission.shape)

display(corpus.head(2))
display(train.head(2))

Corpus: (3600, 5)
Train: (2200, 4)
Test: (800, 2)
Sample submission: (800, 3)


,chunk_id,doc_id,title,doc_type,chunk_text
0,doc_0001_chunk_1,doc_0001,Architecture note: CDN / Data Pipeline / Playback API / Auth,architecture_note,"The streaming platform consists of core services including CDN, Data Pipeline, Playback API, Auth. Requests flow from edge caches to regional clusters with ..."
1,doc_0001_chunk_2,doc_0001,Architecture note: CDN / Data Pipeline / Playback API / Auth,architecture_note,Authentication tokens are validated before playback sessions are authorized. CDN cache rules affect hit ratio and latency during traffic bursts.


,question_id,question,answer,gold_chunk_ids
0,q_0001,"Based on the knowledge base, summarize the key point from: Architecture note: Playback API / Observability / Search / Billing.",Authentication tokens are validated before playback sessions are authorized,['doc_0154_chunk_2']
1,q_0002,Answer using evidence. What is the correct key detail from 'NA data_pipeline_break incident report (2026-06-07)' (ignore unrelated notes)?,"On 2026-06-07, the data_pipeline_break impacted the NA region with severity 4",['doc_1110_chunk_1']


## 3. Validate and prepare the knowledge base

In [4]:
required_corpus_columns = {"chunk_id", "chunk_text"}
required_train_columns = {"question_id", "question", "answer", "gold_chunk_ids"}
required_test_columns = {"question_id", "question"}

assert required_corpus_columns.issubset(corpus.columns)
assert required_train_columns.issubset(train.columns)
assert required_test_columns.issubset(test.columns)

corpus["chunk_id"] = corpus["chunk_id"].astype(str)
corpus["chunk_text"] = corpus["chunk_text"].astype(str)
if "title" not in corpus.columns:
    corpus["title"] = ""

corpus["search_text"] = corpus["title"].astype(str) + " " + corpus["chunk_text"]
print("Knowledge chunks:", len(corpus))
print("Missing values:")
display(corpus[["chunk_id", "chunk_text"]].isna().sum())

Knowledge chunks: 3600
Missing values:


chunk_id      0
chunk_text    0
dtype: int64

## 4. Build the retrieval index

In [5]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=150000
)
chunk_matrix = vectorizer.fit_transform(corpus["search_text"])
print("TF-IDF index shape:", chunk_matrix.shape)

TF-IDF index shape: (3600, 1036)


## 5. Define retrieval and query refinement

In [6]:
def refine_query(question):
    refined = re.sub(
        r"\\b(what is|what are|explain|please|tell me about|how does|how do)\\b",
        " ", str(question), flags=re.I
    )
    refined = re.sub(r"\\s+", " ", refined).strip()
    return refined if refined else str(question)


def retrieve(question, top_k=5):
    query_vector = vectorizer.transform([str(question)])
    scores = cosine_similarity(query_vector, chunk_matrix).ravel()
    indexes = np.argsort(scores)[::-1][:top_k]

    results = corpus.iloc[indexes].copy()
    results["score"] = scores[indexes]
    return results.reset_index(drop=True)


def agent_retrieve(question, top_k=5):
    # Search with the original and refined question.
    first = retrieve(question, top_k=top_k)
    refined_question = refine_query(question)

    if refined_question.lower() != str(question).lower():
        second = retrieve(refined_question, top_k=top_k)
        combined = pd.concat([first, second], ignore_index=True)
        combined = combined.sort_values("score", ascending=False)
        combined = combined.drop_duplicates("chunk_id").head(top_k)
    else:
        combined = first

    # Corrective retry: widen the search if the best score is weak.
    if len(combined) == 0 or float(combined["score"].max()) < 0.01:
        combined = retrieve(question, top_k=top_k * 2).head(top_k)

    return combined.reset_index(drop=True)


example_question = train.iloc[0]["question"]
display(agent_retrieve(example_question)[["chunk_id", "title", "score", "chunk_text"]])

,chunk_id,title,score,chunk_text
0,doc_0154_chunk_1,Architecture note: Playback API / Observability / Search / Billing,0.704106,"The streaming platform consists of core services including Playback API, Observability, Search, Billing. Requests flow from edge caches to regional clusters..."
1,doc_0154_chunk_2,Architecture note: Playback API / Observability / Search / Billing,0.542446,Authentication tokens are validated before playback sessions are authorized. CDN cache rules affect hit ratio and latency during traffic bursts.
2,doc_0154_chunk_3,Architecture note: Playback API / Observability / Search / Billing,0.540079,"Observability combines metrics, traces, and logs to support incident response. Safe rollouts use canaries and staged deployments to prevent widespread regre..."
3,doc_0661_chunk_1,Architecture note: Playback API / Observability / Search / Recommendations,0.537149,"The streaming platform consists of core services including Playback API, Observability, Search, Recommendations. Requests flow from edge caches to regional ..."
4,doc_1158_chunk_1,Architecture note: Playback API / Recommendations / Observability / Search,0.440693,"The streaming platform consists of core services including Playback API, Recommendations, Observability, Search. Requests flow from edge caches to regional ..."


## 6. Define the agent planner and helper tools

In [7]:
def plan_question(question):
    lower = str(question).lower()
    plan = []

    if any(word in lower for word in ["summarize", "summary", "overview"]):
        plan.append("Retrieve evidence and summarize the relevant chunks.")
    elif any(word in lower for word in ["compare", "difference", "versus"]):
        plan.append("Retrieve evidence for the items and compare the findings.")
    elif any(word in lower for word in ["calculate", "compute", "average", "percentage"]):
        plan.append("Retrieve relevant numeric evidence and calculate when possible.")
    elif any(word in lower for word in ["analyze", "analysis", "statistics"]):
        plan.append("Retrieve evidence and inspect the result set.")
    else:
        plan.append("Retrieve the most relevant evidence for the question.")

    plan.append("Check retrieval quality and retry with a refined or wider query if needed.")
    plan.append("Return an answer with source chunk IDs.")
    return plan


def summarize_text(text, max_sentences=5):
    sentences = re.split(r"(?<=[.!?])\\s+", str(text))
    selected = [s.strip() for s in sentences if len(s.strip()) > 20]
    return " ".join(selected[:max_sentences])


def answer_question(question, top_k=5):
    plan = plan_question(question)
    retrieved = agent_retrieve(question, top_k=top_k)
    answer = summarize_text(" ".join(retrieved["chunk_text"].astype(str).tolist()))

    if not answer:
        answer = "I could not find enough information in the knowledge base."

    return {
        "answer": answer,
        "cited_chunk_ids": retrieved["chunk_id"].astype(str).tolist(),
        "plan": plan,
        "retrieved": retrieved
    }


example_result = answer_question(example_question)
print("Question:", example_question)
print("\nPlan:")
for step in example_result["plan"]:
    print("-", step)
print("\nAnswer:", example_result["answer"])
print("\nCitations:", example_result["cited_chunk_ids"])

Question: Based on the knowledge base, summarize the key point from: Architecture note: Playback API / Observability / Search / Billing.

Plan:
- Retrieve evidence and summarize the relevant chunks.
- Check retrieval quality and retry with a refined or wider query if needed.
- Return an answer with source chunk IDs.

Answer: The streaming platform consists of core services including Playback API, Observability, Search, Billing. Requests flow from edge caches to regional clusters with automated health checks. Authentication tokens are validated before playback sessions are authorized. CDN cache rules affect hit ratio and latency during traffic bursts. Observability combines metrics, traces, and logs to support incident response. Safe rollouts use canaries and staged deployments to prevent widespread regressions. The streaming platform consists of core services including Playback API, Observability, Search, Recommendations. Requests flow from edge caches to regional clusters with automat

## 7. Evaluate retrieval on the training set

In [8]:
def parse_id_list(value):
    if isinstance(value, list):
        return [str(item) for item in value]
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
        return [str(parsed)]
    except Exception:
        return []


def evaluate_row(row, k=5):
    retrieved = agent_retrieve(row["question"], top_k=k)
    predicted = retrieved["chunk_id"].astype(str).tolist()
    gold = parse_id_list(row["gold_chunk_ids"])

    if not gold:
        return pd.Series({
            "recall_at_5": 0.0,
            "precision_at_5": 0.0,
            "mrr": 0.0
        })

    hits = [1 if item in gold else 0 for item in predicted]
    recall = len(set(predicted) & set(gold)) / len(set(gold))
    precision = sum(hits) / k

    reciprocal_rank = 0.0
    for rank, hit in enumerate(hits, start=1):
        if hit:
            reciprocal_rank = 1 / rank
            break

    return pd.Series({
        "recall_at_5": recall,
        "precision_at_5": precision,
        "mrr": reciprocal_rank
    })


evaluation = train.apply(evaluate_row, axis=1)
print("Mean retrieval metrics:")
display(evaluation.mean().to_frame("score"))

Mean retrieval metrics:


,score
recall_at_5,0.392500
precision_at_5,0.106636
mrr,0.294545


## 8. Inspect answer grounding and citations

In [9]:
for index in range(min(3, len(train))):
    row = train.iloc[index]
    result = answer_question(row["question"])

    print("=" * 90)
    print("Question:", row["question"])
    print("Reference answer:", row["answer"])
    print("Generated extractive answer:", result["answer"])
    print("Gold chunk IDs:", parse_id_list(row["gold_chunk_ids"]))
    print("Retrieved chunk IDs:", result["cited_chunk_ids"])

Question: Based on the knowledge base, summarize the key point from: Architecture note: Playback API / Observability / Search / Billing.
Reference answer: Authentication tokens are validated before playback sessions are authorized
Generated extractive answer: The streaming platform consists of core services including Playback API, Observability, Search, Billing. Requests flow from edge caches to regional clusters with automated health checks. Authentication tokens are validated before playback sessions are authorized. CDN cache rules affect hit ratio and latency during traffic bursts. Observability combines metrics, traces, and logs to support incident response. Safe rollouts use canaries and staged deployments to prevent widespread regressions. The streaming platform consists of core services including Playback API, Observability, Search, Recommendations. Requests flow from edge caches to regional clusters with automated health checks. The streaming platform consists of core services 

## 9. Create the Kaggle submission

In [10]:
submission = sample_submission[["question_id"]].copy()

answers = []
citations = []

for _, row in test.iterrows():
    result = answer_question(row["question"])
    answers.append(result["answer"])
    citations.append("[" + ", ".join(result["cited_chunk_ids"]) + "]")

submission["answer"] = answers
submission["cited_chunk_ids"] = citations
submission = submission[sample_submission.columns]

submission.to_csv("submission.csv", index=False)
print("Created submission.csv")
display(submission.head())

Created submission.csv


,question_id,answer,cited_chunk_ids
0,q_2201,"On 2025-02-10, the payment_failure impacted the APAC region with severity 4. Customer symptoms included buffering, retries, and elevated error rates. Initia...","[doc_0291_chunk_1, doc_0291_chunk_2, doc_0291_chunk_3, doc_0677_chunk_1, doc_0931_chunk_1]"
1,q_2202,"The streaming platform consists of core services including Playback API, Auth, Search, Recommendations. Requests flow from edge caches to regional clusters ...","[doc_0254_chunk_1, doc_0254_chunk_2, doc_0909_chunk_1, doc_0339_chunk_1, doc_0254_chunk_3]"
2,q_2203,"On 2025-05-11, the latency_spike impacted the EU region with severity 4. Customer symptoms included buffering, retries, and elevated error rates. Initial tr...","[doc_1047_chunk_1, doc_1047_chunk_2, doc_1047_chunk_3, doc_0285_chunk_1, doc_0316_chunk_1]"
3,q_2204,"This policy defines response targets by severity. Severity 5 (global outage): acknowledge within 5 minutes, mitigation within 30 minutes. This policy define...","[doc_0305_chunk_1, doc_0820_chunk_1, doc_0822_chunk_1, doc_0823_chunk_1, doc_0824_chunk_1]"
4,q_2205,"On 2024-05-21, the payment_failure impacted the EU region with severity 4. Customer symptoms included buffering, retries, and elevated error rates. Initial ...","[doc_0071_chunk_1, doc_0071_chunk_2, doc_1189_chunk_1, doc_0071_chunk_3, doc_1189_chunk_2]"


## 10. Download the submission

In [11]:
from IPython.display import FileLink, display
display(FileLink("submission.csv"))

/kaggle/working/submission.csv

## 11. Limitations and next improvements

- This benchmark notebook uses the pre-chunked AgentEval corpus. The companion Streamlit app implements document extraction, cleaning, chunking, and knowledge-base creation from uploaded files.
- The notebook's planner is rule-based and transparent. It is a baseline agent, not an LLM reasoning agent.
- Answers are extractive summaries of retrieved text, not LLM-generated answers.
- The benchmark notebook uses TF-IDF retrieval. Dense embeddings and hybrid retrieval are available in the companion application when the Sentence Transformers model can be downloaded.
- Tool usage in the companion app includes retrieval, summarization, basic arithmetic, and retrieval-result analysis.
- Review the competition's exact `cited_chunk_ids` format before final submission.
- Run the notebook on Kaggle and record the actual evaluation results. No score is assumed in advance.